In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem.Descriptors import MolWt, MolLogP, NumHDonors, NumHAcceptors, TPSA, NumRotatableBonds
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.rdBase import BlockLogs
from chembl_webresource_client.new_client import new_client
from scipy.stats import mannwhitneyu

In [ ]:
# Fetch target NaV1.5 (CHEMBL1980)
target = new_client.target.search("CHEMBL1980")
target_df = pd.DataFrame.from_dict(target)
pro_target_ID = target_df['target_chembl_id'][0]

# Retrieve bioactivity data
bio_activity = new_client.activity.search(pro_target_ID)
bio_activity_df = pd.DataFrame.from_dict(bio_activity)

print(f"Total initial records retrieved: {len(bio_activity_df)}")

In [ ]:
# Filter for IC50 standard type
bio_activity_IC50_df = bio_activity_df[bio_activity_df['standard_type'] == 'IC50']
df = bio_activity_IC50_df[['canonical_smiles', 'molecule_chembl_id', 'assay_description', 'standard_value']].copy()

# Exclude late-phase assays to isolate peak current phase inhibitors
df = df[~df['assay_description'].str.contains('late', case=False, na=False)]

# Drop entries missing SMILES or IC50 values
df = df.dropna(subset=['canonical_smiles', 'standard_value'])
df['standard_value'] = df['standard_value'].astype(float)

print(f"Records after initial filtering: {len(df)}")

In [ ]:
def standardize(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        return None
    
    # Clean, isolate parent fragment, uncharge, and canonicalize tautomers
    clean_mol = rdMolStandardize.Cleanup(mol)
    parent_clean_mol = rdMolStandardize.FragmentParent(clean_mol)
    uncharger = rdMolStandardize.Uncharger()
    uncharged_parent_clean_mol = uncharger.uncharge(parent_clean_mol)
    te = rdMolStandardize.TautomerEnumerator()
    taut_uncharged_parent_clean_mol = te.Canonicalize(uncharged_parent_clean_mol)
    
    return taut_uncharged_parent_clean_mol

# Suppress RDKit logs during standardization
block = BlockLogs()
df['mol'] = df['canonical_smiles'].apply(standardize)
del block

# Drop invalid molecules and generate standard SMILES
df = df.dropna(subset=['mol'])
df['smiles'] = df['mol'].apply(Chem.MolToSmiles)
df = df[['molecule_chembl_id', 'smiles', 'mol', 'standard_value']]

In [ ]:
# Define threshold (nM): 7500 for main study, 5000 for sensitivity analysis
ACTIVITY_THRESHOLD = 7500 

def resolve_duplicates(group):
    """
    If any duplicate has a value below the threshold, retain the most potent one.
    Otherwise, retain the first entry.
    """
    if (group['standard_value'] < ACTIVITY_THRESHOLD).any():
        return group.loc[group['standard_value'].idxmin()]
    else:
        return group.iloc[0]

# Separate duplicated and unique SMILES
mask_duplicate = df.duplicated(['smiles'], keep=False)
df_dupi = df[mask_duplicate].copy()
df_unique = df[~mask_duplicate].copy()

# Resolve duplicates and concatenate
if not df_dupi.empty:
    df_resolved = df_dupi.groupby('smiles').apply(resolve_duplicates, include_groups=False).reset_index()
    df_total = pd.concat([df_unique, df_resolved], axis=0, ignore_index=True)
else:
    df_total = df_unique.reset_index(drop=True)

# Assign class labels
df_total['class'] = df_total['standard_value'].apply(lambda x: 'active' if x <= ACTIVITY_THRESHOLD else 'inactive')

print(f"Total unique records: {len(df_total)}")
print(df_total['class'].value_counts())

In [ ]:
def calc_descriptors(mol):
    if mol:
        Chem.DeleteSubstructs(mol, Chem.MolFromSmarts("[#1X0]"))
        return [MolWt(mol), MolLogP(mol), NumRotatableBonds(mol), NumHDonors(mol), NumHAcceptors(mol), TPSA(mol)]
    return [None] * 6

desc_cols = ['mw', 'logp', 'num_rotable_bounds', 'hbd', 'hba', 'tpsa']
desc_data = df_total['mol'].apply(calc_descriptors).tolist()
desc_df = pd.DataFrame(data=desc_data, columns=desc_cols)

df_total = pd.concat([df_total, desc_df], axis=1)

In [ ]:
def remove_outliers_combined(df, columns):
    mask = pd.Series(True, index=df.index)
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        mask &= (df[col] >= lower_bound) & (df[col] <= upper_bound)
    return df[mask]

clear_dataset = remove_outliers_combined(df_total, desc_cols)
print(f"Records after outlier removal: {len(clear_dataset)}")
print(clear_dataset['class'].value_counts())

In [ ]:
# Visualize feature distributions
rows, cols = 2, 3
fig, axes = plt.subplots(rows, cols, figsize=(10, 14), dpi=150)
title_cols = ['MW Comparison', 'LogP Comparison', 'Rotatable Bonds', 'Num HD', 'Num HA', 'TPSA Comparison'] 

for idx, col in enumerate(desc_cols):
    r, c = divmod(idx, cols)
    sns.boxplot(
        x="class", 
        y=col, 
        data=clear_dataset, 
        showmeans=True, 
        hue="class", 
        ax=axes[r, c]
    )
    axes[r, c].set_title(title_cols[idx], weight='bold', fontsize=14)
    axes[r, c].set_ylabel(col, weight='bold', fontsize=10)

fig.subplots_adjust(wspace=0.5, hspace=0.5)
plt.tight_layout()
fig.savefig('amar-1608.tiff', bbox_inches='tight')
plt.show()

,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,target_organism,target_pref_name,target_tax_id,text_value,toid,type,units,uo_units,upper_value,value
0,None,None,1138475,[],CHEMBL660922,Inhibition of sodium channel blockade in CHO c...,B,None,None,BAO_0000190,...,Homo sapiens,Sodium channel protein type V alpha subunit,9606,None,NaN,IC50,uM,UO_0000065,None,2.85
9,None,None,2629395,[],CHEMBL1001516,Inhibition of Voltage-gated sodium channel sub...,B,None,None,BAO_0000190,...,Homo sapiens,Sodium channel protein type V alpha subunit,9606,None,NaN,IC50,uM,UO_0000065,None,10.0
10,None,None,2629443,[],CHEMBL1001516,Inhibition of Voltage-gated sodium channel sub...,B,None,None,BAO_0000190,...,Homo sapiens,Sodium channel protein type V alpha subunit,9606,None,NaN,IC50,uM,UO_0000065,None,10.0
13,None,None,2679350,[],CHEMBL1053972,Inhibition of Nav1.5 channel,B,None,None,BAO_0000190,...,Homo sapiens,Sodium channel protein type V alpha subunit,9606,None,NaN,IC50,uM,UO_0000065,None,10.0
14,None,None,2679420,[],CHEMBL1053972,Inhibition of Nav1.5 channel,B,None,None,BAO_0000190,...,Homo sapiens,Sodium channel protein type V alpha subunit,9606,None,NaN,IC50,uM,UO_0000065,None,10.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3509,None,None,25081206,[],CHEMBL5255853,Inhibition of recombinant human Nav1.5,T,None,None,BAO_0000190,...,Homo sapiens,Sodium channel protein type V alpha subunit,9606,None,NaN,IC50,uM,UO_0000065,None,50.0
3510,"{'action_type': 'BLOCKER', 'description': 'Neg...",None,25095807,[],CHEMBL5259991,Inhibition of Nav1.5 (unknown origin) by whole...,B,None,None,BAO_0000190,...,Homo sapiens,Sodium channel protein type V alpha subunit,9606,None,NaN,IC50,uM,UO_0000065,None,46.17
3511,None,Not Determined,25095809,[],CHEMBL5259991,Inhibition of Nav1.5 (unknown origin) by whole...,B,None,None,BAO_0000190,...,Homo sapiens,Sodium channel protein type V alpha subunit,9606,None,NaN,IC50,None,None,None,None
3512,"{'action_type': 'BLOCKER', 'description': 'Neg...",None,25095810,[],CHEMBL5259991,Inhibition of Nav1.5 (unknown origin) by whole...,B,None,None,BAO_0000190,...,Homo sapiens,Sodium channel protein type V alpha subunit,9606,None,NaN,IC50,uM,UO_0000065,None,54.72


In [ ]:
# Aggregate descriptive statistics
df_stat = clear_dataset[desc_cols + ['class']].groupby('class').agg(['median', 'mean', 'min', 'max', 'skew']).T
df_stat.to_csv("amar-total-data.csv")

# Perform Mann-Whitney U test across all descriptors
def run_mannwhitney(descriptor):
    active = clear_dataset[clear_dataset['class'] == 'active'][descriptor]
    inactive = clear_dataset[clear_dataset['class'] == 'inactive'][descriptor]
    stat, p = mannwhitneyu(active, inactive)
    
    alpha = 0.05
    interpretation = 'Same distribution (fail to reject H0)' if p > alpha else 'Different distribution (reject H0)'
    
    return pd.DataFrame({
        'Descriptor': [descriptor],
        'Statistics': [stat],
        'p-value': [p],
        'Interpretation': [interpretation]
    })

# Compile and export test results
mw_results = pd.concat([run_mannwhitney(desc) for desc in desc_cols], ignore_index=True)
mw_results.to_csv("mannwhitney_results_combined.csv", index=False)
display(mw_results)